# Classic ML Baselines

This notebook runs the classic baseline grid and creates two analysis tables:

- `total_results`: all evaluated combinations.
- `report_results`: the best validation result per representation family.

The task predicts sentiment labels `0..4` for mixed English/German product reviews. The validation score is `1 - MAE / 4`.

The grid contains a majority baseline; sparse BoW/TF-IDF word/character features with Logistic Regression, Linear SVM, Ridge Classifier, and Complement NB; and optional static embedding features with average, TF-IDF weighted, and mean+max pooling trained with Logistic Regression, Linear SVM, and Ridge Classifier.


In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd

EXPERIMENT_KIND = "DENSE_EMBEDDING_EXPERIMENTS"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/classic_ml") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EMBEDDING_DIR = Path("experiments/embeddings")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
RANDOM_STATE = 42
MAX_FEATURES = 250_000
MAX_ITER = 100

embedding_paths = {
    "glove": EMBEDDING_DIR / "glove.6B.300d.txt",
    "fasttext": EMBEDDING_DIR / "fasttext.vec",
}

available_embeddings = {
    name: path for name, path in embedding_paths.items() if path.exists()
}
available_embeddings

This notebook is configured to run only dense static-embedding experiments. Place optional pretrained static embeddings in `experiments/embeddings/` with the filenames configured above.


In [ ]:
if not available_embeddings:
    raise FileNotFoundError("No embedding files found in experiments/embeddings/. Run load_embeddings.ipynb first.")

mode = "embeddings"

cmd = [
    sys.executable,
    "-m",
    "baselines.classic_ml_baselines",
    "--mode",
    mode,
    "--train-path",
    str(TRAIN_PATH),
    "--output-dir",
    str(EXPERIMENT_DIR),
    "--validation-size",
    str(VALIDATION_SIZE),
    "--random-state",
    str(RANDOM_STATE),
    "--max-features",
    str(MAX_FEATURES),
    "--max-iter",
    str(MAX_ITER),
]

for name, path in available_embeddings.items():
    cmd.extend([f"--{name}-path", str(path)])

print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Total Analysis

In [ ]:
results = pd.read_csv(EXPERIMENT_DIR / "classic_ml_results.csv")

total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results

## Report Analysis

In [ ]:
ok_results = results[results["status"] == "ok"].copy()
report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby("family", as_index=False)
    .first()
    .sort_values("cil_score", ascending=False)
    .reset_index(drop=True)
)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results


## Analysis Artifacts


In [ ]:
analysis_dir = EXPERIMENT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

classic_fastest_table = analysis_dir / "classic_fastest_families.tex"
classic_full_table = analysis_dir / "classic_full_results.tex"
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.make_classic_latex_table",
        "--input",
        str(EXPERIMENT_DIR / "classic_ml_results.csv"),
        "--output",
        str(classic_fastest_table),
        "--full-output",
        str(classic_full_table),
    ],
    check=True,
)

classic_fastest_table, classic_full_table
